In [61]:
import pandas as pd
from pathlib import Path
import os
import sys
import re
import numpy as np

In [4]:
root_path = Path.cwd().resolve().parents[0]
data_path = root_path / 'data' / 'raw'
output_path = root_path / 'data' / 'processed'

In [6]:
inbound = pd.read_csv(data_path / 'total_inbound_tertiary_students_2000-2026.csv')
outbound = pd.read_csv(data_path / 'total_outbound_tertiary_students_2000-2026.csv')

In [7]:
inbound.head()

,Unnamed: 0,indicatorId,geoUnit,year,value,magnitude,qualifier
0,0,26637,ABW,2003,91.0,NaN,NaN
1,1,26637,ABW,2005,40.0,NaN,NaN
2,2,26637,ABW,2006,169.0,NaN,NaN
3,3,26637,ABW,2009,107.0,NaN,NaN
4,4,26637,ABW,2010,63.0,NaN,NaN


In [ ]:
# unstack data to wide to add NaN for missing year values
wide_inbound = inbound.set_index(['geoUnit', 'year']).unstack(level='year')['value']
wide_inbound.head()

year,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
geoUnit,,,,,,,,,,,,,,,,,,,,,
ABW,NaN,NaN,NaN,91.00000,NaN,40.00000,169.000,NaN,NaN,107.0,...,3.290000e+02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AFG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,0.000,NaN,NaN,NaN,NaN,NaN
AIA,NaN,NaN,NaN,4.00000,NaN,NaN,0.000,0.0000,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AIMS: Asia and the Pacific,329030.28125,347121.5,444449.75,479542.46875,477823.1875,510351.15625,540728.625,601349.1875,657290.5,752134.5,...,1.125217e+06,1.205078e+06,1.318228e+06,1456985.875,1511211.375,1449089.00,1471143.875,1567716.875,NaN,NaN
AIMS: Central Asia,21210.00000,16761.0,22318.50,22406.00000,27382.0000,33954.00000,39532.000,42330.0000,39931.0,34376.0,...,2.976420e+04,3.197980e+04,3.396140e+04,46640.000,82775.000,106121.25,124011.500,112958.750,NaN,NaN


In [63]:
# reset data to long for interpolation; interpolation will add
# values forward and between missing data, but not backpropagate to earlier
# NaNs
long_inbound = wide_inbound.reset_index().melt(
    id_vars = 'geoUnit',
    var_name = 'year',
    value_name = 'value'
)
long_inbound['year'] = long_inbound['year'].astype(int)
long_inbound = long_inbound.sort_values(by=['geoUnit', 'year']).reset_index(drop=True)
long_inbound['value'] = long_inbound.groupby('geoUnit')['value'].transform(lambda group: group.interpolate(method= 'linear'))
long_inbound.head(20)

,geoUnit,year,value
0,ABW,2000,NaN
1,ABW,2001,NaN
2,ABW,2002,NaN
3,ABW,2003,91.000000
4,ABW,2004,65.500000
5,ABW,2005,40.000000
6,ABW,2006,169.000000
7,ABW,2007,148.333333
8,ABW,2008,127.666667
9,ABW,2009,107.000000


In [53]:
folder_path = os.path.abspath(os.path.join('..', 'references', 'dictionaries'))
sys.path.append(folder_path)

In [54]:
from country_names import countries

In [ ]:
long_inbound['country'] = [countries.get(country_code, np.nan) for country_code in long_inbound['geoUnit']]

,geoUnit,year,value,country
10265,ZWE,2021,633.0,Zimbabwe
10266,ZWE,2022,633.0,Zimbabwe
10267,ZWE,2023,633.0,Zimbabwe
10268,ZWE,2024,633.0,Zimbabwe
10269,ZWE,2025,633.0,Zimbabwe
